# 04. Адаптивність: діагностика -> вибір обробки, каскад

**Тиждень 3.**
- `classic`: завжди ті самі кроки.
- `adaptive`: спершу `diagnose()` (темно? відблиск? розмито? шум?), потім лише потрібні кроки.
- `cascade`: пробуємо кроки по черзі, поки декодер не прочитає (дорожче за часом).

Пороги діагностики в `src/diagnose.py` (`THRESHOLDS`) поставлено приблизно.
TODO: відкалібрувати їх тут.

In [ ]:
import sys
from pathlib import Path

# ноутбук лежить у notebooks/, корінь проєкту на рівень вище
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)

In [ ]:
from src.data import generate_synthetic
from src.degradations import DEGRADATIONS, LEVELS
from src.diagnose import diagnose, measure, THRESHOLDS
from src.evaluate import run_benchmark, summarize

samples = generate_synthetic(n_per_format=10, seed=42)
THRESHOLDS

## Чи правильно діагностика впізнає проблему?

Для кожного пошкодження очікуємо свій діагноз. `jpeg` і `perspective` діагностика
не розпізнає (для них немає метрики): це свідоме обмеження.

In [ ]:
EXPECTED = {
    "dark": "dark", "uneven": "uneven", "glare": "glare", "noise": "noisy",
    "motion_blur": "blurry", "defocus": "blurry", "jpeg": None, "perspective": None,
}

import itertools

rows = [{"degradation": "clean", "level": 0, "problems": diagnose(s["image"])} for s in samples]
for sample, name, level in itertools.product(samples, DEGRADATIONS, LEVELS):
    damaged = DEGRADATIONS[name](sample["image"], level, seed=level)
    rows.append({"degradation": name, "level": level, "problems": diagnose(damaged)})
diagnosis = pd.DataFrame(rows)

def is_hit(row):
    """1 = діагноз правильний, 0 = ні, NaN = для цього пошкодження метрики немає."""
    if row["degradation"] == "clean":
        return float(len(row["problems"]) == 0)      # чистий кадр: без хибної тривоги
    expected = EXPECTED[row["degradation"]]
    if expected is None:
        return np.nan
    return float(expected in row["problems"])


diagnosis["hit"] = diagnosis.apply(is_hit, axis=1)
print("Частка правильних діагнозів (clean = частка кадрів БЕЗ хибної тривоги):")
diagnosis.dropna(subset=["hit"]).pivot_table(index="degradation", columns="level", values="hit", aggfunc="mean").round(2)

## Метрики кадрів: допомога для калібрування порогів

In [ ]:
rows = [{**measure(s["image"]), "degradation": "clean", "level": 0} for s in samples[:10]]
for sample, name, level in itertools.product(samples[:10], DEGRADATIONS, [1, 3, 5]):
    damaged = DEGRADATIONS[name](sample["image"], level, seed=level)
    rows.append({**measure(damaged), "degradation": name, "level": level})
metric_table = pd.DataFrame(rows)
metric_table.groupby(["degradation", "level"]).median().round(3)

## raw vs classic vs adaptive vs cascade

In [ ]:
table = run_benchmark(samples, methods=["raw", "classic", "adaptive", "cascade"])
table.to_csv(RESULTS / "04_adaptive.csv", index=False)
summarize(table, by=["method", "decoder"])

In [ ]:
summary = summarize(table[table["decoder"] == "zxing"], by=["method", "degradation", "level"])
summary.pivot_table(index=["method", "degradation"], columns="level", values="decode_rate").round(2)

## Каскад: який крок спрацьовує найчастіше?

In [ ]:
cascade_rows = table[(table["method"] == "cascade") & (table["outcome"] == "correct")]
cascade_rows["step"].value_counts()

**TODO для тебе:**
- Відкалібрувати `THRESHOLDS` за таблицею метрик вище (і перевірити, що `clean` лишається без хибних тривог).
- Де `adaptive` гірший за `raw`? Чому? (підказка: `dark` + шум сенсора, `clahe` підсилює шум)
- Ідея: повторна діагностика після кожного кроку.
- Каскад точніший, але повільніший: скільки мс? Чи вкладається в ліміт лінії?